# HDiv-VIM -- curved-geometry accuracy win (showcase)

The FEEC HDiv (Raviart-Thomas) Volume Integral Method for soft-iron
demagnetisation. Its headline advantage over the shipped collocation
surface-charge (six-face) solver is on CURVED geometry: with `mesh.Curve(p)` the
HDiv stray field is far more accurate per element. This is a **conservative
showcase** of that verified win; the full (active) corpus -- the MMMM
multipole-moment line, the streamfunction/cohomology design line, the
conditioning/preconditioner studies, and `reference_yano_msc/` -- is **kept** at
`validation_test/feec/vim_legacy/` for runnable legacy helpers and the VIM retirement ledger for retired prototypes.

Each demo runs on execute and writes its JSON result next to this notebook.
Promoted (showcase) 2026-06-27; source scripts embedded verbatim (paths repointed
to the notebook dir; `radia.vim` is the installed package).

## Nonlinear solver policy (engineering default)

The nonlinear soft-iron demag (`radia.vim.Solve(bh_table=...)`, and the curved x nonlinear
field demo below) follows the lab engineering standard (Sugahara 2026-07-04, at the HDiv-VIM dev close;
memory `feedback-nonlinear-tol-1e3-engineering`):

- **Tolerance <= 1e-3** on `max|dB|/B_sat` per iteration -- do NOT set it tighter for an engineering run
  (the observable carries ~0.5-1% mesh/discretisation error, so 1e-6 is ~3 decades of pure waste = ~2x the
  Picard iterations). Use 1e-6 ONLY to match a validation fixture, and say so in the run metadata.
- **Newton is NOT mandatory.** A **safeguarded Picard** -- Picard + **Anderson(1)** (accept the accelerated
  iterate only when the residual drops; the moment/MMMM path's default) -- is the accepted solver at 1e-3.
  Do not reach for Newton merely to cure a "slow" over-tight (1e-6) run; the correct tolerance fixes that.
- **Honest caveat (measured):** *plain* (unsafeguarded / scalar) Picard **diverges at deep saturation**
  (descending-branch steep slope; `relax_param=0.3` does not rescue) -- exactly the "per-element Picard
  FAILED at deep saturation" failure the shipped damped/energy-Newton was built to solve. So "Picard is
  fine" means **Picard + Anderson(1)**, not naive Picard. The shipped C++ energy-Newton (deep-saturation
  robust) STAYS -- "non-mandatory" is permissive (safeguarded Picard is an accepted alternative at 1e-3),
  NOT "remove the working Newton".

## Elementary curved-geometry win

External (stray) field of a uniformly-magnetised sphere: flat six-face surface-charge vs Curve(3) at the SAME element count. Curved is ~30-40x more accurate vs the analytic dipole truth.

*(source: `validation_test/feec/vim_legacy/hdiv_demag_curved.py`)*

In [1]:
"""hdiv_demag_curved.py -- the CURVED-MESH geometry win for the HDiv-type VIM, measured vs ANALYTIC truth.

HDiv (RT0) lives natively on curved (isoparametric) meshes via mesh.Curve(p) -- the Piola map carries
the curving, and the SAME mesh.GetTrafo code path samples the exact curved surface.  six-face surface-charge flat
ObjHexahedron / ObjTetrahedron CANNOT do this: a flat-faceted sphere has a ~6-10% geometry error at a
coarse mesh that no amount of magnetisation accuracy can recover.  This example MEASURES the win against
ANALYTIC truth (the exact uniform-sphere dipole / volume), NOT against Radia:

  GEOMETRY:  a coarse FLAT sphere has volume ~10% low and area ~6% low; mesh.Curve(3) at the SAME ndof
             fixes both to <0.5% (the isoparametric geometry is exact to the polynomial order).

  EXTERNAL FIELD  (the discriminator):  the external field of a uniform-M sphere is the EXACT point
             dipole  phi(r) = (1/4pi) V cos(theta)/r^2.  This is a surface-charge integral at an
             EXTERNAL point -- no singular quadrature, so the ONLY error is the geometry.  The flat
             coarse mesh gets it ~10% WRONG (it inherits the volume error directly); curved at the same
             ndof is <0.3% -- a 30-40x accuracy win vs analytic truth.

  DEMAG FACTOR  (a CAVEAT about THIS elementary method):  with the crude sub-point Gram used HERE the
             demag FACTOR does NOT cleanly discriminate the curved win -- its ~2% sub-point-quadrature
             BIAS masks the ~0.25% geometry signal (and a coarse polyhedron's demag happens to sit near
             1/3 too).  This is a limitation of the elementary sub-point Gram, NOT a property of the demag
             factor: with the PROPER Gram (the ngsolve.bem Laplace single-layer in
             hdiv_demag_bem_singlelayer.py) the demag factor DOES discriminate cleanly AND p-converges
             (flat floors ~0.25%, curved + order-2 is EXACT).  For THIS self-contained elementary method,
             the EXTERNAL FIELD above is the clean geometry-only discriminator.

Reference (uniform M = z_hat, unit sphere):  V = 4pi/3,  area = 4pi,  D_z = 1/3,
external scalar potential at (0,0,2) = (1/4pi) V / 2^2 = 1/12.

The curved-sampling helper (_trafo_sample) is the building block the accurate curved HDiv-VIM Gram will
reuse (cf. the curved Galerkin single-layer in src/radia/bem/sibc_hacapk.py::_ss_block_curved_trafo).
"""
import json
import os
import sys
from math import pi

import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", "examples", "vim"))
from radia.vim import _core as tet  # noqa: E402  (reuse _bary_tri + C_TRI)

import ngsolve as ng  # noqa: E402
from ngsolve import IntegrationRule, ElementId, BND, CoefficientFunction, Integrate  # noqa: E402
from netgen.csg import CSGeometry, Sphere, Pnt  # noqa: E402

ng.SetNumThreads(4)
HERE = os.getcwd()


def _trafo_sample(mesh, i_bnd, xi, eta, center):
    """Curved physical position / surface Jacobian / OUTWARD unit-normal z-component at ref points
    (xi, eta) on boundary element i_bnd.  Works identically on a linear mesh (affine = flat triangle)
    and a curved mesh (mesh.Curve(p)) -- GetTrafo carries the curving.  This is the reusable HDiv-VIM
    curved-geometry sampling primitive (same pattern as bem/sibc_hacapk.py::_trafo_eval)."""
    Q = len(xi)
    ir = IntegrationRule(points=[(xi[k], eta[k], 0) for k in range(Q)], weights=[1.0] * Q)
    trafo = mesh.GetTrafo(ElementId(BND, i_bnd))
    r = np.zeros((Q, 3)); J = np.zeros(Q); nz = np.zeros(Q)
    for k, ip in enumerate(ir):
        mip = trafo(ip)
        p = np.array([mip.point[0], mip.point[1], mip.point[2]])
        jac = np.asarray(mip.jacobi)                       # (3,2): dr/dxi, dr/deta
        nrm = np.cross(jac[:, 0], jac[:, 1])
        nrm = nrm / (np.linalg.norm(nrm) + 1e-300)
        if np.dot(p - center, nrm) < 0:                    # orient outward via the known center
            nrm = -nrm
        r[k] = p; J[k] = mip.measure; nz[k] = nrm[2]
    return r, J, nz


def _surface_samples(mesh, nsub, center):
    """All boundary sub-points (curved), areas, and normal-z, via _trafo_sample.  w = J * (ref_area)/m."""
    lam = tet._bary_tri(nsub)                              # (m,3) equal-weight barycentric lattice
    xi, eta, m = lam[:, 1], lam[:, 2], len(lam)
    P, W, NZ = [], [], []
    n_bnd = sum(1 for _ in mesh.Elements(BND))
    for i in range(n_bnd):
        r, J, nz = _trafo_sample(mesh, i, xi, eta, center)
        P.append(r); W.append(J * 0.5 / m); NZ.append(nz)
    return np.vstack(P), np.concatenate(W), np.concatenate(NZ), n_bnd


def demag_z_surface(mesh, nsub, center=np.zeros(3)):
    """Demag factor D_z via the uniform-M surface double integral
        D_z = (1/4pi V) INT_S INT_S n_z(r) n_z(r')/|r-r'| dS dS'
    sub-point quadrature + C_TRI sub-cell self.  Identical treatment flat/curved.  NB this crude Gram has
    a ~2% quadrature bias that masks the geometry signal -> use ext_potential (or the ngsolve.bem
    single-layer in hdiv_demag_bem_singlelayer.py) to see the curved win, not this."""
    P, w, nz, n_bnd = _surface_samples(mesh, nsub, center)
    V = float(Integrate(CoefficientFunction(1.0), mesh))
    D = np.linalg.norm(P[:, None, :] - P[None, :, :], axis=2)
    np.fill_diagonal(D, np.inf)
    wn = w * nz
    offdiag = float(np.sum(np.outer(wn, wn) / D))
    selfdiag = float(np.sum(nz ** 2 * tet.C_TRI * w ** 1.5))
    return dict(Dz=(offdiag + selfdiag) / (4 * pi * V), V=V, area=float(w.sum()), n_bnd=n_bnd)


def ext_potential(mesh, nsub, Pobs, center=np.zeros(3)):
    """Surface-charge magnetic scalar potential at an EXTERNAL point (sigma = M n_z, M = 1).  No singular
    quadrature -> the ONLY error is the geometry.  For a uniform unit-M sphere this is the EXACT dipole;
    the flat mesh inherits the volume error (~10% low), curved is exact -> THE curved-win discriminator."""
    P, w, nz, _ = _surface_samples(mesh, nsub, center)
    return float(np.sum(nz * w / np.linalg.norm(P - Pobs, axis=1))) / (4 * pi)


def _sphere(h):
    g = CSGeometry(); g.Add(Sphere(Pnt(0, 0, 0), 1.0))
    return ng.Mesh(g.GenerateMesh(maxh=h))


def run(hs=(0.8, 0.5), curve_order=3):
    V_an, A_an, D_an, phi_an = 4.0 * pi / 3.0, 4.0 * pi, 1.0 / 3.0, 1.0 / 12.0
    Pobs = np.array([0.0, 0.0, 2.0])
    out = {"analytic": {"V": V_an, "area": A_an, "demag_z": D_an, "ext_phi_002": phi_an},
           "curve_order": curve_order, "cases": []}
    for h in hs:
        for curve in (0, curve_order):
            mesh = _sphere(h)
            if curve:
                with ng.TaskManager():
                    mesh.Curve(curve)
            dd = demag_z_surface(mesh, 4)
            phi = ext_potential(mesh, 4, Pobs)
            out["cases"].append(dict(
                h=h, curved=bool(curve), n_bnd=dd["n_bnd"], V=dd["V"], area=dd["area"],
                demag_z=dd["Dz"], ext_phi=phi,
                V_err=dd["V"] / V_an - 1, area_err=dd["area"] / A_an - 1,
                demag_err=dd["Dz"] / D_an - 1, ext_phi_err=phi / phi_an - 1))
    return out


if True:
    res = run()
    print(f"analytic sphere:  V={4*pi/3:.5f}  area={4*pi:.5f}  D_z=1/3  ext_phi(0,0,2)=1/12={1/12:.5f}")
    print("=" * 100)
    print(f"{'h':>5} {'mesh':>8} {'n_bnd':>6} {'V err':>10} {'area err':>10} "
          f"{'D_z':>8} {'D_z err':>9} {'ext field err vs dipole':>24}")
    for c in res["cases"]:
        print(f"{c['h']:>5} {'curve' if c['curved'] else 'FLAT':>8} {c['n_bnd']:>6} "
              f"{100*c['V_err']:>9.2f}% {100*c['area_err']:>9.2f}% {c['demag_z']:>8.4f} "
              f"{100*c['demag_err']:>8.2f}% {100*c['ext_phi_err']:>22.2f}%")
    print("=" * 100)
    print("WIN: curved external field error is ~30-40x smaller than flat at the SAME ndof (vs analytic")
    print("dipole truth).  The demag FACTOR does NOT discriminate (near-isotropic ratio) -- see docstring.")
    with open(os.path.join(HERE, "hdiv_demag_curved.json"), "w") as f:
        json.dump(res, f, indent=2)
    print("saved", os.path.join(HERE, "hdiv_demag_curved.json"))


analytic sphere:  V=4.18879  area=12.56637  D_z=1/3  ext_phi(0,0,2)=1/12=0.08333
    h     mesh  n_bnd      V err   area err      D_z   D_z err  ext field err vs dipole
  0.8     FLAT    108    -10.25%     -5.69%   0.3277    -1.68%                 -10.00%
  0.8    curve    108      0.02%     -0.42%   0.3238    -2.87%                  -0.26%
  0.5     FLAT    126     -8.52%     -4.76%   0.3263    -2.12%                  -8.28%
  0.5    curve    126      0.01%     -0.35%   0.3251    -2.48%                  -0.25%
WIN: curved external field error is ~30-40x smaller than flat at the SAME ndof (vs analytic
dipole truth).  The demag FACTOR does NOT discriminate (near-isotropic ratio) -- see docstring.
saved S:\Radia\01_GitHub\docs\hdiv_vim\hdiv_demag_curved.json


## Curved x nonlinear field win

Stray field of a NONLINEAR soft-iron part: ~9% wrong with flat elements, <0.4% with curved (~23x) -- where curved x nonlinear actually matters (the field, not the bulk magnetisation).

*(source: `validation_test/feec/vim_legacy/hdiv_curved_nonlinear_field.py`)*

In [2]:
"""hdiv_curved_nonlinear_field.py -- (A) the FIELD output of a CURVED x NONLINEAR body: where the curved
win is LARGE (~9x), unlike the magnetization (modest ~0.3%, see test_hdiv_vim_curved_nonlinear.py).

A nonlinear soft-iron sphere in a uniform field magnetizes UNIFORMLY (for ANY M-H law), so M is the
scalar fixed point  M = M(H_ext - D M)  with the curved demag D = 1/3.  The EXTERNAL H field is then the
EXACT point dipole, m = M V.  The curved win lives HERE: the flat faceted sphere's volume is ~9% low, so
its dipole moment -- hence the WHOLE external field -- is ~9% WRONG; mesh.Curve(3) at the SAME ndof is
<0.4% at every external point.  This is the engineering deliverable (the stray field around a nonlinear
soft-iron part) being ~9% off with flat elements (six-face surface-charge) and EXACT with curved (HDiv-VIM).

H(r) = (1/4pi) INT_S sigma(r') (r-r')/|r-r'|^3 dS',  sigma = M.n  (the surface-charge stray field; no
singular quadrature at an EXTERNAL point -> the only error is the geometry).  Validated vs the ANALYTIC
dipole -- Radia cannot referee curved geometry (its ObjHex/Tet facet the body).  The nonlinearity sets
the field MAGNITUDE (physical M); the curved win is the ~9% geometry error, which the nonlinearity does
not amplify (it merely scales it).
"""
import json
import os
import sys
from math import pi

import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", "examples", "vim"))
import hdiv_demag_curved as cv            # _surface_samples (curved-aware surface quadrature)
from radia.vim import _nonlinear as nl     # _scalar_fixed_point

import ngsolve as ng                       # noqa: E402
from ngsolve import TaskManager            # noqa: E402
from netgen.csg import CSGeometry, Sphere, Pnt  # noqa: E402

ng.SetNumThreads(4)
HERE = os.getcwd()

# soft-iron-like saturating law M(H) = chi0 H / (1 + chi0|H|/Msat)
CHI0, MSAT = 5000.0, 1.6e6


def Mof(H):
    return CHI0 * H / (1.0 + CHI0 * abs(H) / MSAT)


def reconstruct_H(mesh, M_scalar, obs, nsub=4):
    """External H field from the surface charge sigma = M_scalar * n_z, curved-aware (the SAME
    mesh.GetTrafo sampling for flat and curved -- only mesh.Curve toggled)."""
    P, w, nz, _ = cv._surface_samples(mesh, nsub, np.zeros(3))
    sw = M_scalar * nz * w                                  # sigma * dS at each surface quad point
    H = np.zeros((len(obs), 3))
    for i, r in enumerate(obs):
        d = r - P
        rn = np.linalg.norm(d, axis=1)
        H[i] = (1.0 / (4 * pi)) * np.sum(sw[:, None] * d / rn[:, None] ** 3, axis=0)
    return H


def H_dipole(r, m):
    """Analytic point-dipole H field, moment m along z."""
    rn = np.linalg.norm(r)
    rh = r / rn
    mv = np.array([0.0, 0.0, m])
    return (1.0 / (4 * pi)) * (3.0 * np.dot(mv, rh) * rh - mv) / rn ** 3


def _sphere(h):
    g = CSGeometry(); g.Add(Sphere(Pnt(0, 0, 0), 1.0))
    return ng.Mesh(g.GenerateMesh(maxh=h))


def run(h=0.6, H_ext=1e4, curve_order=3):
    M_s = nl._scalar_fixed_point(Mof, 1.0 / 3.0, H_ext)    # nonlinear sphere: uniform M, demag D=1/3
    m = M_s * (4.0 * pi / 3.0)                              # dipole moment of the uniform-M sphere
    obs = np.array([[0, 0, 1.5], [0, 0, 2.0], [0, 0, 3.0], [1.5, 0, 0.6], [2.0, 0, 0.0]], float)
    out = {"H_ext": H_ext, "M_s": M_s, "dipole_m": m, "h": h, "obs": obs.tolist(), "cases": []}
    for curve in (0, curve_order):
        mesh = _sphere(h)
        if curve:
            with TaskManager():
                mesh.Curve(curve)
        Hn = reconstruct_H(mesh, M_s, obs)
        errs = [float(np.linalg.norm(Hn[i] - H_dipole(r, m)) / np.linalg.norm(H_dipole(r, m)))
                for i, r in enumerate(obs)]
        out["cases"].append(dict(curved=bool(curve), max_err=max(errs), errs=errs))
    return out


if True:
    res = run()
    print(f"nonlinear soft-iron sphere: M_s = {res['M_s']:.1f} A/m (H_ext={res['H_ext']:.0e}); "
          f"dipole m = {res['dipole_m']:.1f}")
    print("external H-field error vs the ANALYTIC dipole (5 points; geometry-only error):")
    for c in res["cases"]:
        tag = "Curve(3)" if c["curved"] else "FLAT    "
        print(f"  {tag}: max {100*c['max_err']:+.2f}%   per-point " +
              " ".join(f"{100*e:+.2f}%" for e in c["errs"]))
    flat = next(c for c in res["cases"] if not c["curved"])
    curv = next(c for c in res["cases"] if c["curved"])
    print(f"=> the curved field win is ~{flat['max_err']/curv['max_err']:.0f}x: the stray field of a "
          f"nonlinear soft-iron part is ~{100*flat['max_err']:.0f}% wrong with FLAT elements (six-face surface-charge)")
    print("   and <0.4% with curved -- THIS is where curved x nonlinear matters (the field, not M).")
    with open(os.path.join(HERE, "hdiv_curved_nonlinear_field.json"), "w") as f:
        json.dump(res, f, indent=2)
    print("saved", os.path.join(HERE, "hdiv_curved_nonlinear_field.json"))


nonlinear soft-iron sphere: M_s = 29981.7 A/m (H_ext=1e+04); dipole m = 125586.9
external H-field error vs the ANALYTIC dipole (5 points; geometry-only error):
  FLAT    : max +8.85%   per-point +8.85% +8.80% +8.76% +8.54% +8.85%
  Curve(3): max +0.39%   per-point +0.29% +0.34% +0.31% +0.32% +0.39%
=> the curved field win is ~23x: the stray field of a nonlinear soft-iron part is ~9% wrong with FLAT elements (six-face surface-charge)
   and <0.4% with curved -- THIS is where curved x nonlinear matters (the field, not M).


saved S:\Radia\01_GitHub\docs\hdiv_vim\hdiv_curved_nonlinear_field.json


## Head-to-head vs the shipped Radia solver

HDiv curved-coarsest beats flat-finest, i.e. accuracy-per-DOF beats the production surface-charge solver vs the analytic truth (wall-clock is the pending C++ lift).

*(source: `validation_test/feec/vim_legacy/compare_curved_vs_radia_field.py`)*

In [3]:
"""compare_curved_vs_radia_field.py -- (B) head-to-head: external field accuracy-per-resolution,
the SHIPPED Radia solver (FLAT tets) vs HDiv-VIM (CURVED single-layer surface charge), vs the ANALYTIC
dipole of a uniform-M sphere.

This is the quantitative basis for the curved accuracy-per-DOF win against the PRODUCTION code.  Radia's
ObjTetrahedron are FLAT -- the accessible stand-in for the six-face surface-charge distortion elements, which are also
flat.  At the SAME mesh parameter h, the HDiv curved field is ~10-30x more accurate; and curved at the
COARSEST mesh beats shipped-Radia-flat at the FINEST.

HONEST SCOPE: this measures ACCURACY-PER-RESOLUTION (geometry-driven, fair across implementations), NOT
wall-clock.  The HDiv-VIM here is a Python prototype (dense surface-charge sum), not time-optimized; a
fair speed comparison needs the C++ productionization (not done).  Radia cannot referee curved geometry
(it facets), so the reference is the ANALYTIC dipole.  The uniform-M sphere is chosen because its
external field is exactly dipolar (a clean analytic truth); the nonlinearity (soft iron) only scales the
field magnitude by M and does not change this accuracy-per-resolution picture.
"""
import json
import os
import sys
from math import pi

import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", "examples", "vim"))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", "src", "radia"))
import radia as rad                         # the SHIPPED production solver (flat elements)
import ngsolve as ng
from ngsolve import TaskManager
from netgen.csg import CSGeometry, Sphere, Pnt
import netgen_mesh_import as nmi
import hdiv_demag_curved as cv

ng.SetNumThreads(4)
HERE = os.getcwd()

MU0 = 4e-7 * pi
MS = 1.0e5
M_DIP = MS * (4.0 * pi / 3.0)
OBS = np.array([[0, 0, 1.5], [0, 0, 2.0], [0, 0, 3.0], [1.5, 0, 0.6], [2.0, 0, 0.0]], float)


def _Bdip(r):
    rn = np.linalg.norm(r); rh = r / rn; mv = np.array([0.0, 0.0, M_DIP])
    return (MU0 / (4 * pi)) * (3.0 * np.dot(mv, rh) * rh - mv) / rn ** 3


def radia_flat(h):
    """Shipped Radia: flat-tet uniform-M sphere, external B via rad.Fld, max rel error vs dipole."""
    rad.UtiDelAll()
    geo = CSGeometry(); geo.Add(Sphere(Pnt(0, 0, 0), 1.0))
    ngm = ng.Mesh(geo.GenerateMesh(maxh=h))
    ne = int(ngm.ne)
    cont = nmi.netgen_mesh_to_radia(ngm, material={'magnetization': [0, 0, MS]}, units='m', verbose=False)
    B = np.array(rad.Fld(cont, 'b', OBS.tolist())).reshape(-1, 3)
    err = max(np.linalg.norm(B[i] - _Bdip(r)) / np.linalg.norm(_Bdip(r)) for i, r in enumerate(OBS))
    rad.UtiDelAll()
    return ne, float(err)


def hdiv_curved(h, curve=3, nsub=4):
    """HDiv-VIM: curved surface-charge H field (B=mu0 H outside), max rel error vs dipole."""
    geo = CSGeometry(); geo.Add(Sphere(Pnt(0, 0, 0), 1.0))
    mesh = ng.Mesh(geo.GenerateMesh(maxh=h))
    with TaskManager():
        mesh.Curve(curve)
    P, w, nz, nb = cv._surface_samples(mesh, nsub, np.zeros(3))
    sw = MS * nz * w
    err = 0.0
    for r in OBS:
        d = r - P; rn = np.linalg.norm(d, axis=1)
        H = (1.0 / (4 * pi)) * np.sum(sw[:, None] * d / rn[:, None] ** 3, axis=0)
        err = max(err, np.linalg.norm(H - _Bdip(r) / MU0) / np.linalg.norm(_Bdip(r) / MU0))
    return int(nb), float(err)


def run(hs=(0.6, 0.4, 0.3, 0.2)):
    out = {"hs": list(hs), "rows": []}
    for h in hs:
        ne, ef = radia_flat(h)
        nb, ec = hdiv_curved(h)
        out["rows"].append(dict(h=h, radia_tets=ne, radia_flat_err=ef, hdiv_bnd_tris=nb, hdiv_curved_err=ec))
    out["flat_finest_err"] = out["rows"][-1]["radia_flat_err"]
    out["curved_coarsest_err"] = out["rows"][0]["hdiv_curved_err"]
    return out


if True:
    res = run()
    print("external field accuracy vs analytic dipole -- shipped Radia (FLAT) vs HDiv-VIM (CURVED):")
    print(f"  {'h':>5} | {'Radia tets':>10} {'flat err':>10} | {'HDiv tris':>10} {'curved err':>11}")
    for r in res["rows"]:
        print(f"  {r['h']:>5} | {r['radia_tets']:>10d} {100*r['radia_flat_err']:>9.2f}% | "
              f"{r['hdiv_bnd_tris']:>10d} {100*r['hdiv_curved_err']:>10.3f}%")
    fr, cr = res["rows"][-1], res["rows"][0]
    print(f"\n=> shipped Radia FLAT at finest h={fr['h']} ({fr['radia_tets']} tets): "
          f"{100*res['flat_finest_err']:.2f}%  vs  HDiv CURVED at coarsest h={cr['h']} "
          f"({cr['hdiv_bnd_tris']} tris): {100*res['curved_coarsest_err']:.3f}%")
    print("   curved-coarsest beats flat-finest = the accuracy-per-resolution win vs the production")
    print("   solver (vs analytic truth).  ACCURACY-PER-DOF; wall-clock = the C++ lift, not done.")
    with open(os.path.join(HERE, "compare_curved_vs_radia_field.json"), "w") as f:
        json.dump(res, f, indent=2)
    print("saved", os.path.join(HERE, "compare_curved_vs_radia_field.json"))


external field accuracy vs analytic dipole -- shipped Radia (FLAT) vs HDiv-VIM (CURVED):
      h | Radia tets   flat err |  HDiv tris  curved err
    0.6 |        114      8.92% |        120      0.386%
    0.4 |        260      5.87% |        192      0.229%
    0.3 |        477      3.48% |        318      0.147%
    0.2 |       2042      1.71% |        658      0.072%

=> shipped Radia FLAT at finest h=0.2 (2042 tets): 1.71%  vs  HDiv CURVED at coarsest h=0.6 (120 tris): 0.386%
   curved-coarsest beats flat-finest = the accuracy-per-resolution win vs the production
   solver (vs analytic truth).  ACCURACY-PER-DOF; wall-clock = the C++ lift, not done.
saved S:\Radia\01_GitHub\docs\hdiv_vim\compare_curved_vs_radia_field.json
